# Analisis dan Preprocessing Ulasan E-commerce Berbahasa Indonesia

**Nama:** [ISI NAMA LENGKAP]  
**NIM:** [ISI NIM]  

Dataset: [dipawidia/ecommerce-product-reviews-sentiment](https://huggingface.co/datasets/dipawidia/ecommerce-product-reviews-sentiment)

Notebook ini memuat pemeriksaan dataset, distribusi label, preprocessing teks, dan 10 contoh ulasan asli sebelum/sesudah preprocessing.

## 1. Instalasi dan import library
Jalankan sel berikut satu kali pada Google Colab/Jupyter yang memiliki koneksi internet.

In [ ]:
%pip -q install datasets pandas matplotlib seaborn nltk Sastrawi

In [ ]:
import re
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords', quiet=True)
factory = StemmerFactory()
stemmer = factory.create_stemmer()
print('Library siap digunakan')

## 2. Memuat dataset
Dataset pada Hugging Face berbentuk JSON. Kode berikut memuat file JSON mentah melalui `load_dataset` dengan fallback ke `pandas.read_json`.

In [ ]:
DATASET_ID = "dipawidia/ecommerce-product-reviews-sentiment"
JSON_URL = "https://huggingface.co/datasets/dipawidia/ecommerce-product-reviews-sentiment/resolve/main/labeledReview.datasetFix.json"

try:
    ds = load_dataset("json", data_files=JSON_URL, split="train")
    df = ds.to_pandas()
except Exception as e:
    print("Pemuatan melalui Hugging Face Datasets gagal, mencoba pandas:", repr(e))
    df = pd.read_json(JSON_URL)

print("Dataset berhasil dimuat:", df.shape)
display(df.head())

## 3. Informasi dataset: jumlah data, kolom, dan distribusi sentimen

In [ ]:
print(f"Jumlah baris: {len(df):,}")
print(f"Jumlah kolom: {df.shape[1]}")
print("Nama kolom:", df.columns.tolist())
print("\nTipe data:")
print(df.dtypes)

required = {'review', 'sentimen', 'translate'}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Kolom yang diharapkan tidak ditemukan: {missing}. Kolom aktual: {df.columns.tolist()}")

label_counts = df['sentimen'].value_counts(dropna=False).sort_index()
label_table = pd.DataFrame({
    'jumlah': label_counts,
    'persentase': (label_counts / len(df) * 100).round(2)
})
label_table.index.name = 'label_sentimen'
display(label_table)

label_names = {0: 'Negatif (0)', 1: 'Positif (1)'}
plot_data = label_counts.rename(index=label_names)
plt.figure(figsize=(7,4))
sns.barplot(x=plot_data.index.astype(str), y=plot_data.values)
plt.title('Distribusi Label Sentimen')
plt.xlabel('Sentimen')
plt.ylabel('Jumlah ulasan')
plt.tight_layout()
plt.show()

### Interpretasi distribusi
Label `0` merepresentasikan sentimen negatif dan label `1` merepresentasikan sentimen positif. Jumlah dan persentase pada tabel/grafik dihitung dari file dataset yang berhasil dimuat, bukan diperkirakan dari deskripsi dataset.

## 4. Preprocessing teks
Tahapan preprocessing dilakukan secara berurutan: case folding, cleaning, tokenisasi, stopword removal, dan stemming. Stopword Bahasa Indonesia diambil dari NLTK; beberapa kata negasi seperti `tidak`, `bukan`, dan `belum` dipertahankan karena penting untuk makna sentimen.

In [ ]:
# Daftar stopword Bahasa Indonesia
stop_words = set(stopwords.words('indonesian'))
stop_words -= {'tidak', 'bukan', 'belum', 'jangan', 'tanpa'}

def case_folding(text):
    return str(text).lower()

def cleaning(text):
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)  # URL
    text = re.sub(r'\d+', ' ', text)                       # angka
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)               # tanda baca/karakter khusus
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def tokenisasi(text):
    return text.split()

def stopword_removal(tokens):
    return [word for word in tokens if word not in stop_words]

def stemming(tokens):
    return [stemmer.stem(word) for word in tokens]

def preprocess(text):
    text = case_folding(text)
    text = cleaning(text)
    tokens = tokenisasi(text)
    tokens = stopword_removal(tokens)
    tokens = stemming(tokens)
    return ' '.join(tokens)

df['review_preprocessed'] = df['review'].fillna('').apply(preprocess)
display(df[['review', 'review_preprocessed', 'sentimen']].head(10))

## 5. Sepuluh contoh teks sebelum dan sesudah preprocessing
Tabel berikut mengambil 10 baris langsung dari dataset yang dimuat. Hasilnya bukan contoh buatan.

In [ ]:
contoh_10 = df[['review', 'review_preprocessed', 'sentimen']].head(10).copy()
contoh_10.insert(0, 'No', range(1, len(contoh_10) + 1))
pd.set_option('display.max_colwidth', 250)
display(contoh_10)

## 6. Pengaruh preprocessing terhadap kualitas data teks

- **Case folding** menyamakan bentuk huruf sehingga kata dengan kapitalisasi berbeda diperlakukan konsisten.
- **Cleaning** mengurangi noise berupa URL, angka, tanda baca, dan karakter khusus yang tidak dibutuhkan untuk analisis teks.
- **Tokenisasi** memecah kalimat menjadi unit kata agar dapat diproses lebih lanjut.
- **Stopword removal** mengurangi kata umum yang kurang informatif dan dapat membantu menonjolkan kata penting. Kata negasi sengaja dipertahankan karena dapat membalik polaritas sentimen.
- **Stemming** menyatukan variasi imbuhan ke bentuk dasar sehingga variasi kata lebih konsisten.

Preprocessing dapat mengurangi noise dan variasi kosakata, tetapi tidak selalu meningkatkan kualitas untuk semua model. Penghapusan tanda baca/angka atau stemming yang terlalu agresif dapat menghilangkan konteks, intensitas, nama produk, maupun ekspresi sentimen. Karena itu, hasil preprocessing sebaiknya dibandingkan dengan teks asli pada tahap pemodelan.

## 7. Simpan hasil preprocessing
File CSV ini dapat diunduh dari lingkungan notebook.

In [ ]:
OUTPUT_CSV = 'ulasan_preprocessed.csv'
df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
print(f"File tersimpan: {OUTPUT_CSV}")

## Referensi
1. Dipa Widia, [ecommerce-product-reviews-sentiment](https://huggingface.co/datasets/dipawidia/ecommerce-product-reviews-sentiment), Hugging Face Datasets.
2. Sastrawi, library stemming Bahasa Indonesia: https://github.com/har07/PySastrawi
3. NLTK, dokumentasi stopwords: https://www.nltk.org/